# L1d: Building a Floating-Point Report

A `Float64` can be decomposed by hand into its stored fields. In this lab, you will package that procedure as a reusable function.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Turn a procedure into an interface:__ Package the by-hand decomposition of a `Float64` as a documented function that returns each extracted field.
> * __Slice a bit pattern into its fields:__ Extract the sign bit, the eleven biased-exponent bits, and the fifty-two stored fraction bits from a `Float64`, and explain the fixed field widths.
> * __Verify a representation with a round trip:__ Rebuild the original value from the extracted fields and use exact equality to verify the decomposition.

Let's get started!
___

## Setup, Data, and Prerequisites

Begin by including `Include.jl`, which configures the computational environment and loads the resources used in this lab.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. This file sets paths and loads the required external packages. For more information about the functions and types used in this lab, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl"));  # include the Include.jl file

In addition to Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/). `Include.jl` also loads [`src/Compute.jl`](src/Compute.jl), where you will implement the function.

___

## What the function has to do

[The `float64_report(...)` function](src/Compute.jl) takes one `Float64` and returns a named tuple that records its bit fields, local spacing, and reconstructed value.

> __A function as a contract:__
>
> A function's *contract* separates what the caller supplies from what the function promises to return. We will use this input/output view throughout the course: the implementation may change, but any code that honors the contract can rely on the same interface.
>
> __Input__
>
> * `value::Float64`: the one argument supplied by the caller, which is the floating-point number to analyze.
>
> __Output__
>
> A `NamedTuple` with six fields:
>
> * `value::Float64`: the input value, returned unchanged so the report is self-describing.
> * `sign_bit::Char`: the single bit at position 1 of the bit pattern.
> * `exponent_bits::String`: the 11 bits at positions 2 through 12.
> * `fraction_bits::String`: the 52 bits at positions 13 through 64.
> * `spacing::Float64`: the distance to the next representable neighbor, computed with [the `eps(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.eps-Tuple%7BAbstractFloat%7D).
> * `reconstructed::Float64`: the value rebuilt from the sign, exponent, and fraction alone.
>
> For this lab, assume that `value` is an ordinary finite value. Edge cases, such as zeros, extremely small values, infinities, and `NaN`, require additional reconstruction rules.

The reconstructed value provides the central check. If the decomposition is correct, the rebuilt number must equal the original __exactly__, not approximately.

Use exact floating-point equality to test the reconstruction. Reconstruction is a lossless round trip, not a new arithmetic calculation.
___

## Implement the contract

The contract describes what `float64_report(...)` must accept and return. Now we need an algorithm: the sequence of steps the function will use to keep that promise.

The function lives in [`src/Compute.jl`](src/Compute.jl), which contains its signature, documentation, and four `TODO` comments.

> __The algorithm__
>
> 1. Get the 64-character pattern with [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring).
> 2. Slice out the three fields at positions 1, 2:12, and 13:64.
> 3. Convert the extracted bits to numbers and reconstruct $x = S\times\text{significand}\times 2^{E-1023}$, where $S = (-1)^s$, $s$ is the numeric value of `sign_bit`, $E$ is the exponent interpreted as a base-2 integer, and the significand is $1$ plus the weighted fraction bits.
> 4. Return the named tuple, using `eps(value)` for the spacing.

Two Julia details are important for this implementation:

> __Reading bits as numbers:__
>
> * [The `parse(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.parse) converts a string to a number. Its `base` keyword reads binary digits directly: `parse(UInt64, exponent_bits; base = 2)`.
> * Indexing a `String` with a range gives a `String`, but indexing with a single position gives a `Char`. `fraction_bits[k]` is therefore a character, and needs parsing before it can be multiplied.

Complete all four `TODO`s in the file, then restart the kernel and run all cells. The test suite defines the required behavior. An incomplete function raises the expected "not implemented yet" error.

Let's generate a report for `-65.78912`:

In [ ]:
report = float64_report(-65.78912)

The three field widths should be 1, 11, and 52, accounting for all 64 bits of a `Float64`. Let's check these widths and the exact round trip:

In [ ]:
(sign_width = length(string(report.sign_bit)),
 exponent_width = length(report.exponent_bits),
 fraction_width = length(report.fraction_bits),
 total_bits = 1 + length(report.exponent_bits) + length(report.fraction_bits),
 round_trip_is_exact = report.reconstructed == report.value)

A valid report accounts for all 64 bits and reconstructs the original value exactly. Any error in the field boundaries, exponent bias, or implicit leading digit would produce a mismatch.

___

## Spacing and the limits of exact equality

The `spacing` field gives the gap between `value` and the next larger `Float64`. This gap changes with the size of `value`.

Let's calculate the gap for values from `0.1` to `1.0e12`:

In [ ]:
let
    samples = [0.1, 1.0, 100.0, 1.0e6, 1.0e12]
    [(value = v, spacing = float64_report(v).spacing) for v ∈ samples]
end

A `Float64` stores a fixed number of significant bits, not decimal places, so the spacing between representable values grows with magnitude, from about $2\times10^{-16}$ near $1$ to about $10^{-4}$ near $10^{12}$. This limited precision explains why `0.1 + 0.2 == 0.3` is `false`. 

Reconstructing a value from its stored fields is different: it rebuilds the same bit pattern, so exact equality is the correct test.

___

## Test the contract

Let's run the test suite. It checks the field widths, sign bit, spacing, and exact reconstruction of several positive and negative values. All tests should pass.

In [ ]:
let
    @testset verbose = true "CHEME 4/5800 L1d Test Suite" begin

        @testset "field widths" begin
            r = float64_report(1.0)
            @test length(string(r.sign_bit)) == 1
            @test length(r.exponent_bits) == 11
            @test length(r.fraction_bits) == 52
        end

        @testset "sign bit" begin
            @test float64_report(1.0).sign_bit == '0'
            @test float64_report(-1.0).sign_bit == '1'
            @test float64_report(-0.1).sign_bit == '1'
        end

        @testset "exact round trip" begin
            for value ∈ (1.0, -1.0, 0.1, -0.1, 3.1415926535897, -65.78912, 1234.5)
                @test float64_report(value).reconstructed == value
            end
        end

        @testset "spacing" begin
            @test float64_report(1.0).spacing == eps(1.0)
            @test float64_report(1.0e12).spacing > float64_report(1.0).spacing
            @test 0.1 + 0.2 != 0.3
            @test isapprox(0.1 + 0.2, 0.3)
        end
    end
end;

___

## Summary

A `Float64` stores three fields in a fixed-width word. The `float64_report(...)` function provides a reusable way to inspect those fields.

> __Key Takeaways:__
>
> * **A procedure becomes reusable when it becomes an interface:** A documented function packages the slicing and reassembly steps for repeated use.
> * **An exact round trip proves the decomposition:** The value rebuilt from the extracted fields must equal the original value exactly. Exact comparison is correct for this lossless round trip.
> * **Spacing is not constant:** The gap to the next representable value grows with magnitude, which is why a fixed number of significant bits does not mean a fixed number of decimal places, and why computed results need tolerant comparison.

The spacing between representable values is why computed floating-point results often require approximate comparison.
___